# Prediksi TMA — Bengawan Solo (v2: Non-Recursive, CatBoost Ensemble)

Redesign dari pipeline v1 (LightGBM + recursive forecasting + clipping),
mengadopsi prinsip desain dari pipeline teman: **tidak ada fitur lag TMA sama
sekali**, sehingga validasi dan prediksi 8 bulan ke depan melihat informasi
yang identik — tidak ada proses recursive/autoregressive, tidak ada drift
yang perlu ditambal dengan clipping.

**Perubahan kunci dari v1:**
1. **Hapus semua lag TMA** — fitur eksogen non-rekursif saja (cuaca, tanah, iklim, hulu-hilir)
2. **Fix bug data**: sentinel `-999` di `solar_radiation_mj_m2` (100.080 baris, persis di periode test!), dan hapus `rainfall_openmeteo_mm` yang 100% duplikat dari `rainfall_mm`
3. **Fitur hulu-hilir via weighted graph** — SEMUA pos hulu (bukan cuma 1 tetangga terdekat), berbobot `exp(-jarak_hidraulik_km / 75)`, dibangun dari hujan/kelembapan tanah (bukan TMA) — otomatis non-rekursif juga
4. **Normalisasi target robust per-pos**: `z = (TMA - median_pos) / (IQR_pos/1.349)`
5. **Model: CatBoost ensemble 3-seed**, loss Huber (robust ke outlier/spike sensor)
6. **Validasi rolling-origin 4 fold**, horizon panjang (bukan cuma 1 window seperti v1)

**Hasil validasi (RMSE per fold):**

| Fold | Train sampai | Validasi | RMSE |
|---|---|---|---|
| sep_2023 | 18 Sep 2023 | 19 Sep 2023 – 18 Mei 2024 | 2.69 |
| may_2024 | 18 Mei 2024 | 19 Mei 2024 – 18 Jan 2025 | 0.78 |
| sep_2024 | 18 Sep 2024 | 19 Sep 2024 – 18 Mei 2025 | 1.22 |
| jan_2025 | 18 Jan 2025 | 19 Jan 2025 – 18 Sep 2025 | 1.16 |
| **Rata-rata** | | | **1.46** |

`Wonogiri Dam` konsisten jadi pos dengan error terbesar di semua fold —
genuine model error (kemungkinan pola rilis air bendungan berbeda dari
sungai alami), bukan artefak drift.


## 0. Setup

Sesuaikan `DATA_DIR` ke folder dataset kompetisi kamu.


In [1]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from sklearn.metrics import mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', None)

# ---- CONFIG: sesuaikan path ini ----
DATA_DIR = Path('.')              # folder dataset (berisi train.csv, test.csv, data_pendukung/)
OUTPUT_DIR = Path('./output_v2')  # folder untuk simpan file perantara & hasil
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BBOX_BUFFER = 0.3          # buffer (derajat) untuk filter shapefile HydroRIVERS
UPSTREAM_DECAY_KM = 75.0   # decay jarak hidraulik untuk bobot upstream

# Rolling-origin validation folds: (nama, train_sampai, validasi_sampai)
FOLDS = [
    ('sep_2023', '2023-09-19', '2024-05-19'),
    ('may_2024', '2024-05-19', '2025-01-19'),
    ('sep_2024', '2024-09-19', '2025-05-19'),
    ('jan_2025', '2025-01-19', '2025-09-19'),
]
SEEDS = [17, 41, 83]
MIN_LABELS_BEFORE_CUTOFF = 90

# CatBoost hyperparameters (kecilkan ITERATIONS/DEPTH kalau training terlalu lama di mesin kamu)
ITERATIONS = 1000
DEPTH = 6
EARLY_STOPPING = 50
LEARNING_RATE = 0.05

ModuleNotFoundError: No module named 'catboost'

## 1. Feature engineering (non-recursive)

Bangun fitur eksogen lokal (cuaca/tanah/iklim, rolling window 6h-336h),
fitur upstream berbobot (weighted graph traversal dari HydroRIVERS), atribut
statis sungai (catchment area, stream order, dst), dan fitur kalender.
**Tidak ada fitur TMA lag di sini sama sekali.**


In [ ]:
RAIN_COLS = ['rainfall_mm']  # rainfall_openmeteo_mm dibuang: 100% identik dengan rainfall_mm
MEAN_COLS = ['humidity_pct', 'dew_point_c', 'cloud_cover_pct', 'temperature_c',
             'wind_speed_kmh', 'solar_radiation_mj_m2',
             'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm',
             'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa']
CLIMATE_COLS = ['rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34']
ALREADY_AGG_COLS = ['rainfall_max_24h_mm']
STATIC_COLS = ['built_surface_m2', 'landcover_class']
WINDOWS_HOURS = [6, 24, 72, 168, 336]


def build_river_attrs(data_dir, bbox_buffer):
    shp_path = data_dir / 'data_pendukung' / 'HydroRIVERS_v10_au_shp' / 'HydroRIVERS_v10_au.shp'
    koord_path = data_dir / 'data_pendukung' / 'koordinat_pos.csv'

    gdf = gpd.read_file(shp_path)
    koord = pd.read_csv(koord_path)

    b = bbox_buffer
    bbox = box(koord['longitude'].min() - b, koord['latitude'].min() - b,
               koord['longitude'].max() + b, koord['latitude'].max() + b)
    sub = gdf[gdf.intersects(bbox)].copy()

    pts = gpd.GeoDataFrame(koord, geometry=gpd.points_from_xy(koord['longitude'], koord['latitude']), crs='EPSG:4326')
    cols = ['HYRIV_ID', 'NEXT_DOWN', 'MAIN_RIV', 'LENGTH_KM', 'DIST_DN_KM', 'DIST_UP_KM',
            'CATCH_SKM', 'UPLAND_SKM', 'DIS_AV_CMS', 'ORD_STRA', 'ORD_CLAS', 'ORD_FLOW', 'geometry']
    nearest = gpd.sjoin_nearest(pts, sub[cols], how='left', distance_col='dist_deg')
    nearest = nearest.drop_duplicates(subset='nama_pos').reset_index(drop=True)
    nearest['match_dist_km'] = nearest['dist_deg'] * 111.32
    return nearest.drop(columns='geometry')


def build_upstream_weights(river_attrs):
    weights = {}
    for main_riv, g in river_attrs.groupby('MAIN_RIV'):
        g = g.sort_values('DIST_DN_KM').reset_index(drop=True)
        for i in range(len(g)):
            p = g.loc[i, 'nama_pos']
            d_p = g.loc[i, 'DIST_DN_KM']
            ups = g[g['DIST_DN_KM'] > d_p].copy()
            if len(ups) == 0:
                weights[p] = []
                continue
            ups['hyd_dist_km'] = ups['DIST_DN_KM'] - d_p
            ups['raw_w'] = np.exp(-ups['hyd_dist_km'] / UPSTREAM_DECAY_KM)
            ups['norm_w'] = ups['raw_w'] / ups['raw_w'].sum()
            weights[p] = list(zip(ups['nama_pos'], ups['norm_w']))
    return weights


def clean_env(env):
    env = env.copy()
    n_sentinel = (env['solar_radiation_mj_m2'] == -999).sum()
    print(f"  Fixing {n_sentinel} sentinel -999 values in solar_radiation_mj_m2 -> NaN")
    env.loc[env['solar_radiation_mj_m2'] == -999, 'solar_radiation_mj_m2'] = np.nan
    env['wind_dir_sin'] = np.sin(np.deg2rad(env['wind_direction_deg']))
    env['wind_dir_cos'] = np.cos(np.deg2rad(env['wind_direction_deg']))
    return env


def build_local_features(env, targets):
    mean_cols = MEAN_COLS + ['wind_dir_sin', 'wind_dir_cos']
    results = []
    for pos in env['nama_pos'].unique():
        env_p = env[env['nama_pos'] == pos].set_index('datetime').sort_index()
        tgt_p = targets[targets['nama_pos'] == pos].sort_values('datetime')
        if len(tgt_p) == 0:
            continue

        full_idx = pd.date_range(env_p.index.min(), env_p.index.max(), freq='h')
        env_p = env_p.reindex(full_idx)
        cols_to_ffill = mean_cols + RAIN_COLS + CLIMATE_COLS + ALREADY_AGG_COLS
        env_p[cols_to_ffill] = env_p[cols_to_ffill].ffill(limit=31 * 24)
        env_p[STATIC_COLS] = env_p[STATIC_COLS].ffill().bfill()

        feat = pd.DataFrame(index=tgt_p['datetime'].values)
        inst_cols = mean_cols + CLIMATE_COLS + ALREADY_AGG_COLS + STATIC_COLS
        inst = env_p[inst_cols].reindex(feat.index, method='ffill').add_suffix('_now')
        feat = feat.join(inst)

        for w in WINDOWS_HOURS:
            for c in RAIN_COLS:
                s = env_p[c].rolling(f'{w}h', min_periods=1).sum()
                feat[f'{c}_sum_{w}h'] = s.reindex(feat.index, method='ffill').values
            for c in mean_cols:
                s = env_p[c].rolling(f'{w}h', min_periods=1).mean()
                feat[f'{c}_mean_{w}h'] = s.reindex(feat.index, method='ffill').values

        rain_hourly = env_p[RAIN_COLS[0]].fillna(0)
        for hl in [12, 48, 168]:
            api = rain_hourly.ewm(halflife=hl, min_periods=1).mean()
            feat[f'api_hl{hl}h'] = api.reindex(feat.index, method='ffill').values

        feat['nama_pos'] = pos
        feat['datetime'] = feat.index
        results.append(feat.reset_index(drop=True))

    return pd.concat(results, ignore_index=True)


def build_upstream_features(local_feat, upstream_weights):
    up_source_cols = [c for c in local_feat.columns if c not in ('nama_pos', 'datetime') and
                       ('rainfall' in c or 'soil_moisture' in c or 'api_hl' in c)]
    idx_feat = local_feat.set_index(['datetime', 'nama_pos'])[up_source_cols]

    rows = []
    for pos, ups in upstream_weights.items():
        if len(ups) == 0:
            continue
        dt_index = local_feat.loc[local_feat['nama_pos'] == pos, 'datetime'].values
        acc = pd.DataFrame(0.0, index=pd.Index(dt_index, name='datetime'), columns=up_source_cols)
        for up_pos, w in ups:
            try:
                sub = idx_feat.xs(up_pos, level='nama_pos').reindex(dt_index)
            except KeyError:
                continue
            acc = acc.add(sub.values * w, fill_value=0)
        acc = acc.add_prefix('up_')
        acc['nama_pos'] = pos
        acc['datetime'] = dt_index
        rows.append(acc.reset_index(drop=True))

    if not rows:
        return pd.DataFrame(columns=['nama_pos', 'datetime'])
    return pd.concat(rows, ignore_index=True)

In [ ]:
print("Loading train/test ...")
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['datetime'])
test_raw = pd.read_csv(DATA_DIR / 'test.csv')
test = pd.DataFrame()
test['datetime'] = pd.to_datetime(test_raw['id'].str.split(' - ', n=1).str[0])
test['nama_pos'] = test_raw['id'].str.split(' - ', n=1).str[1]
test['id'] = test_raw['id']

targets = pd.concat([
    train[['datetime', 'nama_pos']].assign(split='train'),
    test[['datetime', 'nama_pos']].assign(split='test')
], ignore_index=True)

print("Loading + cleaning data_lingkungan.csv ...")
env = pd.read_csv(DATA_DIR / 'data_pendukung' / 'data_lingkungan.csv', parse_dates=['datetime'])
env = clean_env(env)
env = env.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

print("Building local exogenous features ...")
local_feat = build_local_features(env, targets)
print("  Local feature shape:", local_feat.shape)

print("Building river attributes + upstream weights ...")
river_attrs = build_river_attrs(DATA_DIR, BBOX_BUFFER)
upstream_weights = build_upstream_weights(river_attrs)
n_with_upstream = sum(1 for v in upstream_weights.values() if len(v) > 0)
print(f"  {n_with_upstream}/{len(upstream_weights)} pos have >=1 upstream contributor")

print("Building weighted upstream exogenous features ...")
up_feat = build_upstream_features(local_feat, upstream_weights)
print("  Upstream feature shape:", up_feat.shape)

print("Merging local + upstream + static river attrs + calendar ...")
feat = local_feat.merge(up_feat, on=['nama_pos', 'datetime'], how='left')

static_cols = ['MAIN_RIV', 'LENGTH_KM', 'DIST_DN_KM', 'DIST_UP_KM', 'CATCH_SKM',
               'UPLAND_SKM', 'DIS_AV_CMS', 'ORD_STRA', 'ORD_CLAS', 'ORD_FLOW', 'match_dist_km']
ra = river_attrs[['nama_pos'] + static_cols].copy()
for c in ['CATCH_SKM', 'UPLAND_SKM', 'DIS_AV_CMS', 'LENGTH_KM']:
    ra[f'log1p_{c}'] = np.log1p(ra[c].clip(lower=0))
ra['upstream_confidence'] = np.exp(-ra['match_dist_km'] * 1000 / 1000.0)
feat = feat.merge(ra, on='nama_pos', how='left')

koord = pd.read_csv(DATA_DIR / 'data_pendukung' / 'koordinat_pos.csv')
feat = feat.merge(koord, on='nama_pos', how='left')

dt = feat['datetime']
feat['hour'] = dt.dt.hour
feat['month'] = dt.dt.month
doy = dt.dt.dayofyear
feat['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
feat['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
feat['is_rainy_season'] = feat['month'].isin([11, 12, 1, 2, 3, 4]).astype(int)
feat['nama_pos_cat'] = feat['nama_pos'].astype(str)

feat = feat.merge(targets, on=['nama_pos', 'datetime'], how='left')
feat = feat.merge(test[['datetime', 'nama_pos', 'id']], on=['datetime', 'nama_pos'], how='left')
feat = feat.merge(train[['datetime', 'nama_pos', 'tma_mdpl']], on=['datetime', 'nama_pos'], how='left')

# safety: no leaky feature names
bad_cols = [c for c in feat.columns if c != 'tma_mdpl' and (c.startswith('tma_') or 'target' in c.lower())]
assert not bad_cols, f"Leaky feature names found: {bad_cols}"

train_feat = feat[feat['split'] == 'train'].reset_index(drop=True)
test_feat = feat[feat['split'] == 'test'].reset_index(drop=True)

feature_cols = [c for c in feat.columns if c not in
                ('nama_pos', 'datetime', 'split', 'id', 'tma_mdpl', 'MAIN_RIV')]
cat_features = ['nama_pos_cat']

print("Train features shape:", train_feat.shape)
print("Test features shape:", test_feat.shape)
print("N features:", len(feature_cols))
print("NaN target in train?", train_feat['tma_mdpl'].isna().sum())

train_feat.to_parquet(OUTPUT_DIR / 'train_features.parquet', index=False)
test_feat.to_parquet(OUTPUT_DIR / 'test_features.parquet', index=False)

## 2. Target normalization & model helpers

Normalisasi target robust per-pos, dan fungsi training ensemble CatBoost
(3 seed, loss Huber).


In [ ]:
def fit_target_stats(train_df):
    g = train_df.groupby('nama_pos')['tma_mdpl']
    med = g.median()
    q1, q3 = g.quantile(0.25), g.quantile(0.75)
    iqr = (q3 - q1) / 1.349
    scale = iqr.clip(lower=0.05)
    return pd.DataFrame({'nama_pos': med.index, 'center': med.values, 'scale': scale.values})


def normalize_target(df, stats):
    m = df.merge(stats, on='nama_pos', how='left')
    return (m['tma_mdpl'] - m['center']) / m['scale']


def denormalize(pred_z, nama_pos, stats):
    m = pd.DataFrame({'nama_pos': nama_pos.values}).merge(stats, on='nama_pos', how='left')
    return pred_z * m['scale'].values + m['center'].values


def train_ensemble(X_tr, y_tr, cat_features, X_va=None, y_va=None,
                    iterations=ITERATIONS, depth=DEPTH, early_stopping=EARLY_STOPPING):
    models = []
    for seed in SEEDS:
        model = CatBoostRegressor(
            loss_function='Huber:delta=1.5', iterations=iterations, learning_rate=LEARNING_RATE,
            depth=depth, random_seed=seed, cat_features=cat_features,
            early_stopping_rounds=early_stopping if X_va is not None else None,
            verbose=False,
        )
        eval_set = Pool(X_va, y_va, cat_features=cat_features) if X_va is not None else None
        model.fit(X_tr, y_tr, cat_features=cat_features, eval_set=eval_set, use_best_model=(eval_set is not None))
        models.append(model)
    return models


def predict_ensemble(models, X):
    return np.mean([m.predict(X) for m in models], axis=0)

## 3. Validasi rolling-origin (4 fold, horizon panjang)

Setiap fold hanya melatih data sebelum cutoff dan menguji seluruh rentang
berikutnya (horizon panjang, mendekati 8 bulan asli). Pos dengan histori
kurang dari 90 label sebelum cutoff tidak dinilai pada fold tersebut.

**Catatan waktu:** setiap fold melatih 3 model CatBoost (3 seed) — bisa
memakan waktu beberapa menit per fold tergantung mesin kamu. Total ~4 fold
x beberapa menit.


In [ ]:
fold_metrics = []
for fold_name, cutoff, val_end in FOLDS:
    cutoff_ts = pd.Timestamp(cutoff)
    val_end_ts = pd.Timestamp(val_end)

    tr_fold = train_feat[train_feat['datetime'] < cutoff_ts]
    va_fold = train_feat[(train_feat['datetime'] >= cutoff_ts) & (train_feat['datetime'] < val_end_ts)]

    counts = tr_fold.groupby('nama_pos').size()
    valid_pos = counts[counts >= MIN_LABELS_BEFORE_CUTOFF].index
    va_fold_eval = va_fold[va_fold['nama_pos'].isin(valid_pos)]

    if len(tr_fold) == 0 or len(va_fold_eval) == 0:
        print(f"[{fold_name}] skipped (insufficient data)")
        continue

    stats = fit_target_stats(tr_fold)
    y_tr = normalize_target(tr_fold, stats)
    y_va = normalize_target(va_fold_eval, stats)
    X_tr = tr_fold[feature_cols]
    X_va = va_fold_eval[feature_cols]

    models = train_ensemble(X_tr, y_tr, cat_features, X_va, y_va)
    pred_z = predict_ensemble(models, X_va)
    pred = denormalize(pred_z, va_fold_eval['nama_pos'], stats)
    truth = va_fold_eval['tma_mdpl'].values

    rmse = np.sqrt(mean_squared_error(truth, pred))
    mae = mean_absolute_error(truth, pred)
    per_pos_mae = pd.DataFrame({'nama_pos': va_fold_eval['nama_pos'].values,
                                 'abs_err': np.abs(truth - pred)}).groupby('nama_pos')['abs_err'].mean()
    macro_mae = per_pos_mae.mean()
    worst_mae = per_pos_mae.max()
    best_iters = [m.get_best_iteration() or ITERATIONS for m in models]

    print(f"[{fold_name}] n_val={len(va_fold_eval)} RMSE={rmse:.4f} MAE={mae:.4f} "
          f"macroMAE={macro_mae:.4f} worstMAE={worst_mae:.4f} (worst pos: {per_pos_mae.idxmax()}) "
          f"best_iters={best_iters}")
    fold_metrics.append({'fold': fold_name, 'rmse': rmse, 'mae': mae,
                          'macro_mae': macro_mae, 'worst_mae': worst_mae,
                          'best_iteration': int(np.median(best_iters))})

metrics_df = pd.DataFrame(fold_metrics)
metrics_df.to_csv(OUTPUT_DIR / 'cv_metrics.csv', index=False)
print("\n=== CV summary ===")
print(metrics_df.to_string())
print(f"\nMean RMSE across folds: {metrics_df['rmse'].mean():.4f}")
print(f"Mean macro-MAE across folds: {metrics_df['macro_mae'].mean():.4f}")

recommended_iters = int(metrics_df['best_iteration'].median())
print(f"\nRecommended final iterations (median best_iteration across folds): {recommended_iters}")

## 4. Training final + submission

Retrain ensemble di SELURUH data train (tanpa holdout), pakai jumlah
iterasi = median `best_iteration` dari 4 fold di atas. Karena tidak ada
fitur rekursif, prediksi test bisa langsung dilakukan sekali jalan (tidak
perlu loop step-by-step seperti pipeline v1).


In [ ]:
stats_final = fit_target_stats(train_feat)
stats_final.to_csv(OUTPUT_DIR / 'target_stats_final.csv', index=False)
y_all = normalize_target(train_feat, stats_final)
X_all = train_feat[feature_cols]

final_models = train_ensemble(X_all, y_all, cat_features, iterations=recommended_iters)
for i, m in enumerate(final_models):
    m.save_model(str(OUTPUT_DIR / f'catboost_seed_{SEEDS[i]}.cbm'))

X_test = test_feat[feature_cols]
pred_z_test = predict_ensemble(final_models, X_test)
pred_test = denormalize(pred_z_test, test_feat['nama_pos'], stats_final)

submission = test_feat[['id']].copy()
submission['tma_mdpl'] = pred_test
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

print("Saved submission.csv, shape:", submission.shape)
print(submission.head())
print(submission['tma_mdpl'].describe())

## 5. Verifikasi submission.csv

Cek format, kecocokan `id`, dan kewajaran nilai per pos sebelum disubmit ke Kaggle.


In [ ]:
sub = pd.read_csv(OUTPUT_DIR / 'submission.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
train_check = pd.read_csv(DATA_DIR / 'train.csv')

print("Shape:", sub.shape, "(harus: (21780, 2))")
print("NaN:", sub['tma_mdpl'].isna().sum(), "| Duplikat id:", sub['id'].duplicated().sum())
print("Urutan id cocok dg sample_submission?", (sub['id'].values == sample['id'].values).all())

sub['nama_pos'] = sub['id'].str.split(' - ', n=1).str[1]
train_range = train_check.groupby('nama_pos')['tma_mdpl'].agg(['min', 'max'])
sub_range = sub.groupby('nama_pos')['tma_mdpl'].agg(['min', 'max'])
compare = train_range.join(sub_range, lsuffix='_train', rsuffix='_sub')
compare['out_of_range'] = (compare['min_sub'] < compare['min_train'] - 0.5) | \
                          (compare['max_sub'] > compare['max_train'] * 1.3 + 0.5)
print("\nPos di luar batas wajar historis:", compare['out_of_range'].sum())
print(compare[compare['out_of_range']])

## 6. Kesimpulan & ide pengembangan lanjutan

**Yang sudah dikerjakan (v2):**
- Feature engineering non-recursive sepenuhnya (tanpa lag TMA)
- Fix bug data: sentinel `-999` pada `solar_radiation_mj_m2`, hapus fitur duplikat
- Fitur upstream via weighted graph traversal (semua pos hulu, bukan cuma 1)
- Normalisasi target robust per-pos
- CatBoost ensemble 3-seed dengan loss Huber
- Validasi rolling-origin 4 fold horizon panjang

**Hasil:** RMSE rata-rata 4 fold ~1.46, jauh lebih stabil dibanding pipeline
v1 (yang sempat RMSE 12+ tanpa clipping di window musim hujan).

**Yang masih perlu diperbaiki:**
`Wonogiri Dam` konsisten jadi pos dengan error terbesar di semua fold —
genuine model error, bukan artefak drift. Kemungkinan perlu fitur khusus
(misal aturan operasional bendungan, atau data debit rilis) untuk
menangkap pola yang tidak murni hidrologis alami.

**Ide lanjutan lainnya:**
- Hyperparameter tuning CatBoost lebih serius (Optuna)
- Coba iterations/depth lebih besar kalau waktu training memungkinkan
- Analisis residual per pos/bulan/intensitas hujan untuk pola sistematis
- Bandingkan ensemble CatBoost vs LightGBM vs blend keduanya


In [ ]:
import sys, subprocess
print('executable:', sys.executable)
try:
    import catboost
    print('catboost', catboost.__version__)
except Exception as e:
    print('import error:', e)
    print('Installing catboost==1.2.10 in kernel...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost==1.2.10'])
    import importlib
    catboost = importlib.import_module('catboost')
    print('installed catboost', catboost.__version__)
